# R4 · Chạy ma trận thí nghiệm LLM (`src/run_llm.py`)

Notebook này giúp bạn chỉ cần **chọn cấu hình và bấm Run** trên Colab/Kaggle hoặc GPU cá nhân.

Ma trận thí nghiệm R4 (Plan4project C4):
- `Qwen/Qwen3.5-2B` × `{vi_test, en_test}` × `{light, heavy}` × `{basic, strict}`
- Tùy chọn (nếu dư thời gian ở Tuần 2): `Qwen/Qwen3.5-4B`

Script `src/run_llm.py` tự động:
- Tách output theo mức nhiễu `light` / `heavy` vào `results/predictions/`
- Lưu log `format_error`, `has_think`, tốc độ vào `results/logs/`
- Hỗ trợ **resume** (chạy lại không mất các dòng đã xong nếu bị ngắt session).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Chuẩn bị môi trường
- Nếu chạy trên **GPU cá nhân** tại thư mục repo `e:/fin-ocrfix`, giữ nguyên `REPO_DIR = "."`.
- Nếu chạy trên **Colab/Kaggle**, nhớ kéo dữ liệu `data/*.jsonl` và `prompts/*.txt` vào repo trước (hoặc mount Google Drive để không mất file predictions khi hết session).

In [ ]:
import os, pathlib, sys

# Tự nhận diện đang ở thư mục gốc repo hay trong notebooks/
if pathlib.Path("src/run_llm.py").exists():
    REPO_DIR = pathlib.Path(".").resolve()
elif pathlib.Path("../src/run_llm.py").exists():
    REPO_DIR = pathlib.Path("..").resolve()
else:
    REPO_DIR = pathlib.Path("fin-ocrfix").resolve()

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
assert pathlib.Path("src/run_llm.py").exists(), "Không tìm thấy src/run_llm.py — hãy kiểm tra lại thư mục repo."

In [ ]:
# Chạy unit test CPU trước khi đốt GPU
!python -m unittest discover -s tests -v

## 2. Chạy thử trên tập `dev` (Tuần 1, ngày 3–7)
Chạy sau khi R1 đã đẩy `data/vi_dev.jsonl` và `data/en_dev.jsonl` (và đã cập nhật 3 ví dụ few-shot thật trong `prompts/*.txt`).

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-2B"
BATCH_SIZE = 4   # T4 16GB: thử 4 hoặc 8; GPU 8GB: thử 2 hoặc 4
DTYPE = "auto"   # auto | fp16 | bf16 | fp32

DEV_JOBS = [
    ("data/vi_dev.jsonl", "prompts/vi_basic.txt"),
    ("data/vi_dev.jsonl", "prompts/vi_strict.txt"),
    ("data/en_dev.jsonl", "prompts/en_basic.txt"),
    ("data/en_dev.jsonl", "prompts/en_strict.txt"),
]

for data_path, prompt_path in DEV_JOBS:
    if not pathlib.Path(data_path).exists():
        print(f"[Bỏ qua] Chưa có {data_path}")
        continue
    cmd = (
        f"{sys.executable} src/run_llm.py "
        f"--model {MODEL_ID} --data {data_path} --prompt {prompt_path} "
        f"--out_dir results/predictions_dev --log_dir results/logs_dev "
        f"--batch_size {BATCH_SIZE} --dtype {DTYPE}"
    )
    print(">>>", cmd)
    ret = os.system(cmd)
    assert ret == 0, f"Lỗi khi chạy: {cmd}"

## 3. Chạy FULL ma trận tập `test` (Tuần 2, ngày 1–4)
Chạy đủ `vi_test.jsonl` & `en_test.jsonl` × `{light, heavy}` × `{basic, strict}`.
File đầu ra nằm ở `results/predictions/qwen3.5-2b-{basic|strict}_{vi|en}_{light|heavy}.jsonl`.

In [ ]:
TEST_JOBS = [
    ("data/vi_test.jsonl", "prompts/vi_basic.txt"),
    ("data/vi_test.jsonl", "prompts/vi_strict.txt"),
    ("data/en_test.jsonl", "prompts/en_basic.txt"),
    ("data/en_test.jsonl", "prompts/en_strict.txt"),
]

for data_path, prompt_path in TEST_JOBS:
    if not pathlib.Path(data_path).exists():
        print(f"[Bỏ qua] Chưa có {data_path}")
        continue
    cmd = (
        f"{sys.executable} src/run_llm.py "
        f"--model {MODEL_ID} --data {data_path} --prompt {prompt_path} "
        f"--batch_size {BATCH_SIZE} --dtype {DTYPE}"
    )
    print(">>>", cmd)
    ret = os.system(cmd)
    assert ret == 0, f"Lỗi khi chạy: {cmd}"

## 4. (Tùy chọn) Chạy thêm `Qwen/Qwen3.5-4B` nếu dư thời gian ở Tuần 2
Lưu ý mở model card của `Qwen/Qwen3.5-4B` trước để xác nhận cách tắt thinking.

In [ ]:
RUN_4B = False  # Đổi thành True nếu muốn chạy thêm bản 4B
if RUN_4B:
    for data_path, prompt_path in TEST_JOBS:
        if not pathlib.Path(data_path).exists():
            continue
        cmd = (
            f"{sys.executable} src/run_llm.py "
            f"--model Qwen/Qwen3.5-4B --data {data_path} --prompt {prompt_path} "
            f"--batch_size 2 --dtype {DTYPE}"
        )
        print(">>>", cmd)
        assert os.system(cmd) == 0

## 5. Tổng hợp tỷ lệ lỗi định dạng & `<think>` (phục vụ báo cáo RQ4)

In [ ]:
import json, pandas as pd

log_files = sorted(pathlib.Path("results/logs").glob("*.meta.json"))
if not log_files:
    print("Chưa có file log nào trong results/logs/.")
else:
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in log_files]
    cols = ["model_id", "prompt_variant", "lang", "noise", "total_lines", "format_error_count", "format_error_rate", "think_marker_count", "empty_prediction_count", "sec_per_line"]
    df = pd.DataFrame(rows)[cols]
    display(df)